In [ ]:
import os, json, random, time, zipfile, inspect
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple, Set

import numpy as np
import torch
from torch.utils.data import Dataset, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from transformers.trainer_callback import EarlyStoppingCallback

# Colab download helper (safe to import; try/except used later)
try:
    from google.colab import files
    _IN_COLAB = True
except Exception:
    files = None
    _IN_COLAB = False


# -------------------------
# USER CONTROLS
# -------------------------
DOWNLOAD_RESULTS_JSON = True          # download results JSON after final eval
DOWNLOAD_FINAL_MODEL_ZIP = False      # optional: model zips can be large


# ============================================================
# YOUR JSON KEYS
# ============================================================
DOC_ID_KEY = "record_id"
TOK_KEY    = "tokens"
TAG_KEY    = "labels"


# ============================================================
# GLOBALS (set after label inference from TRAIN)
# ============================================================
ENTITY_TYPES: List[str] = []
BIO_LABELS: List[str] = []
LABEL2ID: Dict[str, int] = {}
ID2LABEL: Dict[int, str] = {}


# ============================================================
# Reproducibility
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# HF compatibility helpers
# ============================================================
def _make_training_args(**kwargs) -> TrainingArguments:
    """
    Use eval_strategy everywhere in this notebook.
    This helper remaps eval_strategy <-> evaluation_strategy for older/newer versions.
    """
    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters.keys()); valid.discard("self")

    if "evaluation_strategy" in valid and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    if "eval_strategy" in valid and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

    filtered = {k: v for k, v in kwargs.items() if k in valid}
    return TrainingArguments(**filtered)

def _trainer_tokenizer_kw(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    return {"tokenizer": tokenizer} if "tokenizer" in sig.parameters else {}

def _extract_logits(predictions):
    if isinstance(predictions, (tuple, list)):
        return predictions[0]
    return predictions


# ============================================================
# Safe divisions
# ============================================================
def _safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def _f1(p, r):
    return (2*p*r/(p+r)) if (p+r) else 0.0


# ============================================================
# Doc container
# ============================================================
@dataclass
class DocExample:
    doc_id: str
    words: List[str]
    labels: List[str]     # repaired/normalised gold used for training/scoring
    labels_raw: List[str] # raw gold from file (for reporting/ablation)


# ============================================================
# JSON loading
# ============================================================
def load_json_array(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not isinstance(obj, list):
        raise ValueError(f"Expected list JSON at {path}, got {type(obj)}")
    return obj

def load_raw_records(path: str) -> List[Dict[str, Any]]:
    arr = load_json_array(path)
    out = []
    for i, x in enumerate(arr):
        if DOC_ID_KEY not in x:
            raise ValueError(f"[{path}] record {i} missing DOC_ID_KEY='{DOC_ID_KEY}'. keys={list(x.keys())}")
        if TOK_KEY not in x:
            raise ValueError(f"[{path}] record {i} missing TOK_KEY='{TOK_KEY}'. keys={list(x.keys())}")
        if TAG_KEY not in x:
            raise ValueError(f"[{path}] record {i} missing TAG_KEY='{TAG_KEY}'. keys={list(x.keys())}")

        doc_id = str(x[DOC_ID_KEY])
        words  = list(x[TOK_KEY])
        tags   = x[TAG_KEY]

        if tags is None:
            raise ValueError(
                f"[{path}] record {i} has TAG_KEY='{TAG_KEY}' = None for doc_id={doc_id}. "
                f"If this is a 'test' file with missing labels, use the GOLD eval file here."
            )
        tags = list(tags)

        if len(words) != len(tags):
            raise ValueError(f"[{path}] record {i} len mismatch doc={doc_id}: words={len(words)} tags={len(tags)}")

        out.append({"id": doc_id, "words": words, "tags": tags})
    return out


# ============================================================
# Infer entity types from TRAIN raw tags
# ============================================================
def infer_entity_types_from_records(records: List[Dict[str, Any]]) -> List[str]:
    types = set()
    for r in records:
        for t in r["tags"]:
            if t == "O" or t is None:
                continue
            if "-" not in t:
                continue
            pref, et = t.split("-", 1)
            if pref in ("B", "I") and et and et != "O":
                types.add(et)
    return sorted(types)

def set_label_schema(entity_types: List[str]):
    global ENTITY_TYPES, BIO_LABELS, LABEL2ID, ID2LABEL
    ENTITY_TYPES = list(entity_types)
    BIO_LABELS = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ["B", "I"]]
    LABEL2ID = {l:i for i,l in enumerate(BIO_LABELS)}
    ID2LABEL = {i:l for l,i in LABEL2ID.items()}


# ============================================================
# BIO repair (defensive)
# ============================================================
def repair_bio_sequence(tags: List[str]) -> Tuple[List[str], int]:
    changed = 0
    out = []
    prev_type = "O"

    for t in tags:
        if t in ("B-O", "I-O"):
            out.append("O"); changed += 1
            prev_type = "O"
            continue

        if t == "O":
            out.append("O"); prev_type = "O"
            continue

        if "-" not in t:
            out.append("O"); changed += 1
            prev_type = "O"
            continue

        pref, et = t.split("-", 1)
        lab = f"{pref}-{et}"
        if lab not in LABEL2ID:
            out.append("O"); changed += 1
            prev_type = "O"
            continue

        if pref == "B":
            out.append(lab)
            prev_type = et
        elif pref == "I":
            if prev_type == et:
                out.append(lab)
            else:
                out.append(f"B-{et}")
                changed += 1
                prev_type = et
        else:
            out.append("O"); changed += 1
            prev_type = "O"

    return out, changed


# ============================================================
# Convert raw records -> DocExample with repair policy
# ============================================================
def records_to_docs(records: List[Dict[str, Any]], do_repair: bool) -> Tuple[List[DocExample], Dict[str, Any]]:
    changed_total = 0
    total_tags = 0
    docs = []
    for r in records:
        doc_id = r["id"]
        words = r["words"]
        tags_raw = r["tags"]
        tags = tags_raw[:]
        if do_repair:
            tags, ch = repair_bio_sequence(tags)
            changed_total += ch
        total_tags += len(tags)
        docs.append(DocExample(doc_id=doc_id, words=words, labels=tags, labels_raw=tags_raw))
    rep = {"changed": int(changed_total), "total": int(total_tags), "changed_pct": float(changed_total/(total_tags+1e-12))}
    return docs, rep


# ============================================================
# Load train+eval; infer label schema from TRAIN
# ============================================================
def load_i2b2_train_and_eval(train_path: str, eval_path: str,
                            repair_train_gold: bool = True, repair_eval_gold: bool = False) -> Tuple[List[DocExample], List[DocExample], Dict[str, Any]]:
    train_rec = load_raw_records(train_path)
    eval_rec  = load_raw_records(eval_path)

    inferred = infer_entity_types_from_records(train_rec)
    if not inferred:
        raise ValueError("No entity types inferred from TRAIN. Expected BIO labels like B-XXX/I-XXX.")

    set_label_schema(inferred)

    train_docs, rep_tr = records_to_docs(train_rec, do_repair=repair_train_gold)
    eval_docs,  rep_ev = records_to_docs(eval_rec,  do_repair=repair_eval_gold)

    tr_ids = set(d.doc_id for d in train_docs)
    ev_ids = set(d.doc_id for d in eval_docs)
    overlap = len(tr_ids & ev_ids)
    if overlap != 0:
        raise ValueError(f"Doc ID overlap train∩eval={overlap} (must be 0)")

    rep = {
        "inferred_entity_types": inferred,
        "repair_train_gold": bool(repair_train_gold),
        "repair_eval_gold": bool(repair_eval_gold),
        "train_changed": rep_tr["changed"], "train_total": rep_tr["total"], "train_changed_pct": rep_tr["changed_pct"],
        "eval_changed": rep_ev["changed"],  "eval_total": rep_ev["total"],  "eval_changed_pct": rep_ev["changed_pct"],
        "train_len": len(train_docs),
        "eval_len": len(eval_docs),
        "id_overlap_train_eval": int(overlap),
    }
    return train_docs, eval_docs, rep


# ============================================================
# Tokenizer builder (RoBERTa add_prefix_space fix)
# ============================================================
def build_tokenizer(model_id: str):
    kwargs = {}
    if "roberta" in model_id.lower():
        kwargs["add_prefix_space"] = True
    return AutoTokenizer.from_pretrained(model_id, use_fast=True, **kwargs)


# ============================================================
# Sliding window dataset (NO metadata returned; word_ids cached)
# ============================================================
class SlidingWindowDataset(Dataset):
    def __init__(self,
                 docs: List[DocExample],
                 tokenizer,
                 max_len: int,
                 stride: int,
                 compute_sample_weights: bool = False,
                 window_boosts: Optional[Dict[str, float]] = None):
        self.docs = docs
        self.tok = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.compute_sample_weights = compute_sample_weights
        self.window_boosts = window_boosts or {}

        self.windows = []  # {doc_i, start, end}
        for di, d in enumerate(docs):
            n = len(d.words)
            if n == 0:
                continue
            win_words = max(32, max_len // 2)
            step = max(1, win_words - max(1, stride//2))
            s = 0
            while s < n:
                e = min(n, s + win_words)
                self.windows.append({"doc_i": di, "start": s, "end": e})
                if e == n:
                    break
                s += step

        # cache word_ids for merge (do not batch)
        self._word_ids_cache: List[Optional[List[Optional[int]]]] = [None] * len(self.windows)

        self.sample_weights = None
        if self.compute_sample_weights:
            self.sample_weights = []
            for w in self.windows:
                d = self.docs[w["doc_i"]]
                seg = d.labels[w["start"]:w["end"]]
                wgt = 1.0
                for t in seg:
                    if t == "O":
                        continue
                    et = t.split("-", 1)[1]
                    wgt *= float(self.window_boosts.get(et, 1.0))
                wgt = min(10.0, max(0.1, wgt))
                self.sample_weights.append(wgt)

    def __len__(self):
        return len(self.windows)

    def _encode_window(self, idx: int):
        w = self.windows[idx]
        d = self.docs[w["doc_i"]]
        start, end = w["start"], w["end"]
        words = d.words[start:end]
        tags  = d.labels[start:end]

        enc = self.tok(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
        )
        word_ids = enc.word_ids()
        self._word_ids_cache[idx] = word_ids

        labels = []
        prev_wid = None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev_wid:
                labels.append(LABEL2ID.get(tags[wid], LABEL2ID["O"]))
            else:
                labels.append(-100)
            prev_wid = wid

        return enc, labels

    def get_padded_word_ids(self, idx: int, target_len: int) -> List[Optional[int]]:
        if self._word_ids_cache[idx] is None:
            _enc, _labels = self._encode_window(idx)
        wid = self._word_ids_cache[idx]
        assert wid is not None
        if len(wid) < target_len:
            return wid + [None] * (target_len - len(wid))
        return wid[:target_len]

    def __getitem__(self, idx: int):
        enc, labels = self._encode_window(idx)
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


# ============================================================
# Optional class weights
# ============================================================
def compute_class_weights_from_docs(docs: List[DocExample],
                                   o_mult=0.7, clip_min=0.2, clip_max=5.0,
                                   **type_mults) -> torch.Tensor:
    type_counts = {t:0 for t in ENTITY_TYPES}
    for d in docs:
        for tag in d.labels:
            if tag == "O":
                continue
            et = tag.split("-",1)[1]
            if et in type_counts:
                type_counts[et] += 1
    total = sum(type_counts.values()) + 1e-12
    base = {et: (total/(c+1e-12)) for et,c in type_counts.items()}

    w = np.ones(len(BIO_LABELS), dtype=np.float32)
    w[LABEL2ID["O"]] = float(o_mult)

    for et in ENTITY_TYPES:
        mult = float(type_mults.get(et, 1.0))
        v = float(np.clip(base.get(et, 1.0) * mult, clip_min, clip_max))
        for pref in ("B","I"):
            lab = f"{pref}-{et}"
            if lab in LABEL2ID:
                w[LABEL2ID[lab]] = v

    return torch.tensor(w, dtype=torch.float)


# ============================================================
# CustomTrainer: compatible with num_items_in_batch (Transformers>=4.4x)
# ============================================================
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs.logits

        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device),
                ignore_index=-100
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)

        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


# ============================================================
# Merge window logits back to doc-level predictions
# ============================================================
def merge_logits_to_docs(dataset: SlidingWindowDataset,
                         logits: np.ndarray,
                         label_ids: np.ndarray) -> Tuple[List[str], List[List[str]], List[List[str]]]:
    doc_sums = []
    doc_cnts = []
    doc_gold = []
    for d in dataset.docs:
        n = len(d.words)
        doc_sums.append(np.zeros((n, len(BIO_LABELS)), dtype=np.float64))
        doc_cnts.append(np.zeros((n,), dtype=np.float64))
        doc_gold.append(d.labels)

    for i in range(len(dataset)):
        meta = dataset.windows[i]
        di = meta["doc_i"]
        start = meta["start"]

        li = logits[i]
        labs = label_ids[i]
        target_len = labs.shape[0]

        word_ids = dataset.get_padded_word_ids(i, target_len)

        for pos, wid in enumerate(word_ids):
            if wid is None:
                continue
            if labs[pos] == -100:
                continue
            abs_w = start + wid
            if 0 <= abs_w < doc_sums[di].shape[0]:
                doc_sums[di][abs_w] += li[pos]
                doc_cnts[di][abs_w] += 1.0

    yt_docs, yp_docs, doc_order = [], [], []
    for di, d in enumerate(dataset.docs):
        n = len(d.words)
        if n == 0:
            continue
        avg = doc_sums[di] / np.maximum(doc_cnts[di][:, None], 1.0)
        pred_ids = avg.argmax(axis=-1).tolist()
        pred_tags = [ID2LABEL[int(pid)] for pid in pred_ids]
        yt_docs.append(doc_gold[di])
        yp_docs.append(pred_tags)
        doc_order.append(d.doc_id)

    return doc_order, yt_docs, yp_docs


# ============================================================
# Metrics: Token-level typed micro-F1 (BIO collapsed to type)
# ============================================================
def _to_type(tag: str) -> str:
    if tag == "O": return "O"
    if "-" not in tag: return "O"
    return tag.split("-", 1)[1]

def token_typed_breakdown(yt_docs: List[List[str]], yp_docs: List[List[str]]) -> Dict[str, Any]:
    tp = fp = fn = 0
    per = {e: {"tp":0,"fp":0,"fn":0,"gold":0,"pred":0} for e in ENTITY_TYPES}

    for yt, yp in zip(yt_docs, yp_docs):
        for g, p in zip(yt, yp):
            gt = _to_type(g)
            pt = _to_type(p)

            if gt != "O" and gt in per: per[gt]["gold"] += 1
            if pt != "O" and pt in per: per[pt]["pred"] += 1

            if gt == pt and gt != "O":
                tp += 1
                per[gt]["tp"] += 1
            else:
                if pt != "O":
                    fp += 1
                    per[pt]["fp"] += 1
                if gt != "O":
                    fn += 1
                    per[gt]["fn"] += 1

    P = _safe_div(tp, tp+fp)
    R = _safe_div(tp, tp+fn)
    F = _f1(P, R)

    per_out, macro_parts = {}, []
    for e, c in per.items():
        ep = _safe_div(c["tp"], c["tp"]+c["fp"])
        er = _safe_div(c["tp"], c["tp"]+c["fn"])
        ef = _f1(ep, er)
        macro_parts.append(ef)
        gold, pred = c["gold"], c["pred"]
        per_out[e] = {
            **{k:int(v) for k,v in c.items()},
            "bias": int(pred-gold),
            "bias_pct": float((pred-gold)/(gold+1e-12)),
            "precision": float(ep),
            "recall": float(er),
            "f1": float(ef),
        }
    macro_f1 = float(sum(macro_parts)/len(macro_parts)) if macro_parts else 0.0

    return {
        "overall": {"precision": float(P), "recall": float(R), "f1": float(F), "macro_f1": float(macro_f1),
                    "tp": int(tp), "fp": int(fp), "fn": int(fn)},
        "per_entity": per_out
    }


# ============================================================
# Metrics: Strict entity exact-match micro-F1 (typed)
# ============================================================
def extract_entities_bio(tags: List[str]) -> List[Tuple[int, int, str]]:
    tags_rep, _ = repair_bio_sequence(tags)
    spans = []
    cur_type = None
    start = None

    def close(i):
        nonlocal cur_type, start
        if cur_type is not None and start is not None:
            spans.append((start, i, cur_type))
        cur_type, start = None, None

    for i, t in enumerate(tags_rep):
        if t == "O":
            close(i); continue
        if "-" not in t:
            close(i); continue
        pref, et = t.split("-", 1)
        if pref == "B":
            close(i)
            cur_type, start = et, i
        elif pref == "I":
            if cur_type == et and start is not None:
                continue
            close(i)
            cur_type, start = et, i
        else:
            close(i)

    close(len(tags_rep))
    return spans

def entity_exact_breakdown(yt_docs: List[List[str]], yp_docs: List[List[str]]) -> Dict[str, Any]:
    tp = fp = fn = 0
    per = {e: {"tp":0,"fp":0,"fn":0,"gold":0,"pred":0} for e in ENTITY_TYPES}

    for yt, yp in zip(yt_docs, yp_docs):
        gold_spans = extract_entities_bio(yt)
        pred_spans = extract_entities_bio(yp)
        gold_set: Set[Tuple[int,int,str]] = set(gold_spans)
        pred_set: Set[Tuple[int,int,str]] = set(pred_spans)

        inter = gold_set & pred_set
        tp += len(inter)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        for (s,e,t) in gold_spans:
            if t in per: per[t]["gold"] += 1
        for (s,e,t) in pred_spans:
            if t in per: per[t]["pred"] += 1
        for (s,e,t) in inter:
            if t in per: per[t]["tp"] += 1
        for (s,e,t) in (pred_set - gold_set):
            if t in per: per[t]["fp"] += 1
        for (s,e,t) in (gold_set - pred_set):
            if t in per: per[t]["fn"] += 1

    P = _safe_div(tp, tp+fp)
    R = _safe_div(tp, tp+fn)
    F = _f1(P, R)

    per_out, macro_parts = {}, []
    for e, c in per.items():
        ep = _safe_div(c["tp"], c["tp"]+c["fp"])
        er = _safe_div(c["tp"], c["tp"]+c["fn"])
        ef = _f1(ep, er)
        macro_parts.append(ef)
        gold, pred = c["gold"], c["pred"]
        per_out[e] = {
            **{k:int(v) for k,v in c.items()},
            "bias": int(pred-gold),
            "bias_pct": float((pred-gold)/(gold+1e-12)),
            "precision": float(ep),
            "recall": float(er),
            "f1": float(ef),
        }
    macro_f1 = float(sum(macro_parts)/len(macro_parts)) if macro_parts else 0.0

    return {
        "overall": {"precision": float(P), "recall": float(R), "f1": float(F), "macro_f1": float(macro_f1),
                    "tp": int(tp), "fp": int(fp), "fn": int(fn)},
        "per_entity": per_out
    }

def score_docs_by_mode(yt_docs, yp_docs, mode: str) -> Dict[str, Any]:
    if mode == "token_typed":
        return token_typed_breakdown(yt_docs, yp_docs)
    if mode == "entity_exact":
        return entity_exact_breakdown(yt_docs, yp_docs)
    raise ValueError(f"Unknown mode: {mode}")


# ============================================================
# CV folds and inner-val split (doc-level)
# ============================================================
def build_folds_doclevel(n_docs: int, n_folds: int, seed: int) -> List[List[int]]:
    idx = list(range(n_docs))
    random.Random(seed).shuffle(idx)
    folds = [[] for _ in range(n_folds)]
    for i, v in enumerate(idx):
        folds[i % n_folds].append(v)
    return folds

def split_inner_val_doclevel(train_fold_docs: List[DocExample], val_frac: float, seed: int) -> Tuple[List[DocExample], List[DocExample]]:
    n = len(train_fold_docs)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(round(val_frac * n)))
    val_idx = set(idx[:n_val])
    tr_in = [d for i,d in enumerate(train_fold_docs) if i not in val_idx]
    va_in = [d for i,d in enumerate(train_fold_docs) if i in val_idx]
    return tr_in, va_in


# ============================================================
# Model config
# ============================================================
@dataclass
class ModelCfg:
    name: str
    model_id: str
    max_len: int = 512
    stride: int = 256

    batch_size: int = 4
    grad_accum: int = 2
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    lr_scheduler_type: str = "linear"
    max_grad_norm: float = 1.0
    fp16: bool = True

    epochs_cv: int = 8
    epochs_final: int = 10

    n_folds: int = 5
    inner_val_frac: float = 0.10

    window_boosts: Optional[Dict[str, float]] = None

    use_class_weights: bool = False
    class_weight_multipliers: Optional[Dict[str, float]] = None

    early_stop_patience: int = 3
    early_stop_threshold: float = 0.0

    repair_train_gold: bool = True
    repair_eval_gold: bool = False


def _cfg_defaults(cfg: ModelCfg):
    if cfg.window_boosts is None:
        cfg.window_boosts = {"LOCATION": 1.3, "PHONE": 1.0}
    if cfg.class_weight_multipliers is None:
        cfg.class_weight_multipliers = {"LOCATION": 1.5, "PHONE": 1.1}
    return cfg


# ============================================================
# Train one fold (nested selection + one-shot fold test)
# ============================================================
def train_one_fold_nested(cfg: ModelCfg,
                          tokenizer,
                          train_fold_docs: List[DocExample],
                          test_fold_docs:  List[DocExample],
                          out_dir: str,
                          seed: int,
                          mode: str) -> Dict[str, Any]:
    os.makedirs(out_dir, exist_ok=True)
    cfg = _cfg_defaults(cfg)

    tr_in, va_in = split_inner_val_doclevel(train_fold_docs, cfg.inner_val_frac, seed + 1337)

    train_ds = SlidingWindowDataset(tr_in, tokenizer, cfg.max_len, cfg.stride,
                                    compute_sample_weights=True, window_boosts=cfg.window_boosts)
    val_ds   = SlidingWindowDataset(va_in, tokenizer, cfg.max_len, cfg.stride,
                                    compute_sample_weights=False)
    test_ds  = SlidingWindowDataset(test_fold_docs, tokenizer, cfg.max_len, cfg.stride,
                                    compute_sample_weights=False)

    sampler = None
    if train_ds.sample_weights is not None:
        weights = torch.tensor(train_ds.sample_weights, dtype=torch.double)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    collator = DataCollatorForTokenClassification(tokenizer)
    tkw = _trainer_tokenizer_kw(tokenizer)

    config = AutoConfig.from_pretrained(cfg.model_id, num_labels=len(BIO_LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    model = AutoModelForTokenClassification.from_pretrained(cfg.model_id, config=config)

    class_weights = None
    if cfg.use_class_weights:
        mults = dict(cfg.class_weight_multipliers or {})
        class_weights = compute_class_weights_from_docs(tr_in, o_mult=0.7, clip_min=0.2, clip_max=5.0, **mults)

    args = _make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs_cv,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,
        max_grad_norm=cfg.max_grad_norm,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,      # selects using INNER-VAL only
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=seed,
        report_to="none",
        save_total_limit=1,
        logging_strategy="epoch",
        fp16=(cfg.fp16 and torch.cuda.is_available()),
    )

    val_raw_map = {d.doc_id: d.labels_raw for d in va_in}

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        logits = _extract_logits(logits)
        doc_order, yt, yp = merge_logits_to_docs(val_ds, logits, labels)

        main = score_docs_by_mode(yt, yp, mode)
        yt_raw = [val_raw_map[doc_id] for doc_id in doc_order]
        raw  = score_docs_by_mode(yt_raw, yp, mode)

        return {
            "precision": main["overall"]["precision"],
            "recall": main["overall"]["recall"],
            "f1": main["overall"]["f1"],
            "macro_f1": main["overall"]["macro_f1"],
            "f1_raw_gold": raw["overall"]["f1"],
        }

    callbacks = [EarlyStoppingCallback(
        early_stopping_patience=cfg.early_stop_patience,
        early_stopping_threshold=cfg.early_stop_threshold
    )]

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
        class_weights=class_weights,
        **tkw
    )

    if sampler is not None:
        def _get_train_dataloader_override():
            return torch.utils.data.DataLoader(
                train_ds,
                batch_size=cfg.batch_size,
                sampler=sampler,
                collate_fn=collator,
            )
        trainer.get_train_dataloader = _get_train_dataloader_override

    t0 = time.time()
    trainer.train()
    train_seconds = float(time.time() - t0)
    best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)

    # One-shot fold test (NO selection)
    test_raw_map = {d.doc_id: d.labels_raw for d in test_fold_docs}
    preds = trainer.predict(test_ds)
    logits = _extract_logits(preds.predictions)
    doc_order, yt, yp = merge_logits_to_docs(test_ds, logits, preds.label_ids)

    main = score_docs_by_mode(yt, yp, mode)
    yt_raw = [test_raw_map[doc_id] for doc_id in doc_order]
    raw  = score_docs_by_mode(yt_raw, yp, mode)

    w = np.array(train_ds.sample_weights, dtype=np.float64) if train_ds.sample_weights is not None else np.array([1.0])
    w_summary = {
        "min": float(w.min()), "max": float(w.max()), "mean": float(w.mean()),
        "pct_boosted": float((w > 1.0).mean()),
        "window_boosts": cfg.window_boosts
    }

    return {
        "best_checkpoint": best_ckpt,
        "train_seconds": train_seconds,
        "mode": mode,
        "train_fold_docs": len(train_fold_docs),
        "train_inner_docs": len(tr_in),
        "val_inner_docs": len(va_in),
        "test_fold_docs": len(test_fold_docs),
        "window_sampling": w_summary,
        "test_main": main,
        "test_raw_gold": raw,
    }


# ============================================================
# Final locked eval training (one-shot eval; no best-model-on-eval)
# ============================================================
def train_final_locked(cfg: ModelCfg,
                       tokenizer,
                       train_docs: List[DocExample],
                       eval_docs:  List[DocExample],
                       out_dir: str,
                       seed: int,
                       mode: str) -> Dict[str, Any]:
    os.makedirs(out_dir, exist_ok=True)
    cfg = _cfg_defaults(cfg)

    train_ds = SlidingWindowDataset(train_docs, tokenizer, cfg.max_len, cfg.stride,
                                    compute_sample_weights=True, window_boosts=cfg.window_boosts)
    eval_ds  = SlidingWindowDataset(eval_docs, tokenizer, cfg.max_len, cfg.stride,
                                    compute_sample_weights=False)

    sampler = None
    if train_ds.sample_weights is not None:
        weights = torch.tensor(train_ds.sample_weights, dtype=torch.double)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    collator = DataCollatorForTokenClassification(tokenizer)
    tkw = _trainer_tokenizer_kw(tokenizer)

    config = AutoConfig.from_pretrained(cfg.model_id, num_labels=len(BIO_LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    model = AutoModelForTokenClassification.from_pretrained(cfg.model_id, config=config)

    class_weights = None
    if cfg.use_class_weights:
        mults = dict(cfg.class_weight_multipliers or {})
        class_weights = compute_class_weights_from_docs(train_docs, o_mult=0.7, clip_min=0.2, clip_max=5.0, **mults)

    args = _make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs_final,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,
        max_grad_norm=cfg.max_grad_norm,
        eval_strategy="no",
        save_strategy="epoch",
        load_best_model_at_end=False,      # critical: no selection on eval
        seed=seed,
        report_to="none",
        save_total_limit=1,
        logging_strategy="epoch",
        fp16=(cfg.fp16 and torch.cuda.is_available()),
    )

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=None,
        data_collator=collator,
        compute_metrics=None,
        class_weights=class_weights,
        **tkw
    )

    if sampler is not None:
        def _get_train_dataloader_override():
            return torch.utils.data.DataLoader(
                train_ds,
                batch_size=cfg.batch_size,
                sampler=sampler,
                collate_fn=collator,
            )
        trainer.get_train_dataloader = _get_train_dataloader_override

    t0 = time.time()
    trainer.train()
    train_seconds = float(time.time() - t0)

    # ONE-SHOT eval
    preds = trainer.predict(eval_ds)
    logits = _extract_logits(preds.predictions)
    doc_order, yt, yp = merge_logits_to_docs(eval_ds, logits, preds.label_ids)

    main = score_docs_by_mode(yt, yp, mode)
    raw  = score_docs_by_mode([d.labels_raw for d in eval_docs], yp, mode)

    final_model_dir = os.path.join(out_dir, "final_model")
    trainer.save_model(final_model_dir)

    return {
        "train_seconds": train_seconds,
        "final_model_dir": final_model_dir,
        "eval_main": main,
        "eval_raw_gold": raw,
        "eval_doc_order": doc_order,
    }


def zip_dir(src_dir: str, out_zip: str):
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files_ in os.walk(src_dir):
            for fn in files_:
                p = os.path.join(root, fn)
                rel = os.path.relpath(p, src_dir)
                z.write(p, rel)


# ============================================================
# Unified runner (+ auto-download at end)
# ============================================================
def run_full_pipeline(cfg: ModelCfg,
                      mode: str,
                      seed: int = 42,
                      train_json: str = "/content/train.json",
                      eval_json: str  = "/content/eval.json") -> str:
    set_seed(seed)
    cfg = _cfg_defaults(cfg)

    train_docs, eval_docs, rep = load_i2b2_train_and_eval(
        train_json, eval_json,
        repair_train_gold=cfg.repair_train_gold,
        repair_eval_gold=cfg.repair_eval_gold
    )

    tokenizer = build_tokenizer(cfg.model_id)

    tag = "TOKEN_TYPED" if mode == "token_typed" else "ENTITY_EXACT"
    out_base = f"/content/outputs/{cfg.name.lower()}_{tag}"
    os.makedirs(out_base, exist_ok=True)

    # Nested CV
    folds = build_folds_doclevel(len(train_docs), cfg.n_folds, seed)
    fold_rows = []
    for fi, test_idx in enumerate(folds, start=1):
        test_set = set(test_idx)
        train_fold = [d for i, d in enumerate(train_docs) if i not in test_set]
        test_fold  = [d for i, d in enumerate(train_docs) if i in test_set]

        fold_dir = os.path.join(out_base, f"fold_{fi}")
        print(f"\n[{cfg.name}][{tag}][fold {fi}] train_fold={len(train_fold)} test_fold={len(test_fold)}")
        res = train_one_fold_nested(cfg, tokenizer, train_fold, test_fold, fold_dir, seed + fi, mode=mode)
        res["fold"] = fi
        fold_rows.append(res)
        o = res["test_main"]["overall"]
        print(f"[{cfg.name}][{tag}][fold {fi}] one-shot test F1={o['f1']:.6f} P={o['precision']:.6f} R={o['recall']:.6f}")

    # Final locked eval
    final_dir = os.path.join(out_base, "final_locked_eval")
    final = train_final_locked(cfg, tokenizer, train_docs, eval_docs, final_dir, seed, mode=mode)
    eo = final["eval_main"]["overall"]
    print(f"\n[{cfg.name}][{tag}][FINAL locked eval] F1={eo['f1']:.6f} P={eo['precision']:.6f} R={eo['recall']:.6f}")

    metric_def = (
        "token-level typed micro-F1 (BIO collapsed; wrong type counts as FP+FN)"
        if mode == "token_typed"
        else "STRICT entity-level exact-match micro-F1 (typed; span boundaries must match exactly)"
    )

    payload = {
        "schema_version": "i2b2_2006_papergrade_v2",
        "metric_definition": metric_def,
        "seed": seed,
        "paths": {"train_json": train_json, "eval_json": eval_json},
        "repair_report": rep,
        "model_cfg": asdict(cfg),
        "cv_folds_nested": fold_rows,
        "final_locked_eval": final,
        "lock_eval_policy": "Nested CV selection on inner-val only; eval is one-shot and never used for checkpoint selection.",
    }

    out_json = os.path.join(out_base, f"{cfg.name.lower()}_{tag.lower()}_results.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    model_zip = os.path.join(out_base, f"{cfg.name.lower()}_final_model.zip")
    zip_dir(final["final_model_dir"], model_zip)

    print("Saved results JSON:", out_json)
    print("Zipped final model:", model_zip)

    # ---- AUTO-DOWNLOAD AFTER FINAL EVAL ----
    if DOWNLOAD_RESULTS_JSON or DOWNLOAD_FINAL_MODEL_ZIP:
        try:
            if files is None:
                raise RuntimeError("google.colab.files not available (not running in Colab UI).")
            if DOWNLOAD_RESULTS_JSON:
                files.download(out_json)
            if DOWNLOAD_FINAL_MODEL_ZIP:
                files.download(model_zip)
        except Exception as e:
            print("Auto-download skipped:", repr(e))

    return out_json


# ============================================================
# Convenience wrappers (your call style)
# ============================================================
def run_full_pipeline_entity_exact(cfg, seed=42, train_json="/content/train.json", eval_json="/content/eval.json"):
    return run_full_pipeline(cfg, mode="entity_exact", seed=seed, train_json=train_json, eval_json=eval_json)

def run_full_pipeline_token_typed(cfg, seed=42, train_json="/content/train.json", eval_json="/content/eval.json"):
    return run_full_pipeline(cfg, mode="token_typed", seed=seed, train_json=train_json, eval_json=eval_json)


# ============================================================
# 3 MODEL CONFIGS
# ============================================================
SEED = 42

clinical_cfg = ModelCfg(
    name="ClinicalBERT",
    model_id="emilyalsentzer/Bio_ClinicalBERT",
    max_len=512, stride=256,
    batch_size=4, grad_accum=2, lr=3e-5,
    epochs_cv=8, epochs_final=10,
    window_boosts={"LOCATION": 1.3, "PHONE": 1.0},
    use_class_weights=True,
    class_weight_multipliers={"LOCATION": 1.5, "PHONE": 1.1},
)

biobert_cfg = ModelCfg(
    name="BioBERT",
    model_id="dmis-lab/biobert-base-cased-v1.1",
    max_len=512, stride=256,
    batch_size=4, grad_accum=2, lr=3e-5,
    epochs_cv=8, epochs_final=10,
    window_boosts={"LOCATION": 1.3, "PHONE": 1.0},
    use_class_weights=True,
    class_weight_multipliers={"LOCATION": 1.5, "PHONE": 1.1},
)

roberta_cfg = ModelCfg(
    name="RoBERTaLarge",
    model_id="roberta-large",
    max_len=512, stride=256,
    batch_size=2, grad_accum=4, lr=2e-5,
    epochs_cv=8, epochs_final=10,
    window_boosts={"LOCATION": 1.3, "PHONE": 1.0},
    use_class_weights=True,
)

print("Setup complete.")
print("JSON keys:", DOC_ID_KEY, TOK_KEY, TAG_KEY)
print("Auto-download settings:", {"DOWNLOAD_RESULTS_JSON": DOWNLOAD_RESULTS_JSON, "DOWNLOAD_FINAL_MODEL_ZIP": DOWNLOAD_FINAL_MODEL_ZIP})
print("\nRun one at a time, e.g.:")
print("  out_json = run_full_pipeline_token_typed(clinical_cfg, seed=SEED)")
print("  out_json = run_full_pipeline_token_typed(biobert_cfg, seed=SEED)")
print("  out_json = run_full_pipeline_token_typed(roberta_cfg, seed=SEED)")
print("\nOr strict entity exact-match:")
print("  out_json = run_full_pipeline_entity_exact(clinical_cfg, seed=SEED)")


In [ ]:
SEED = 42
out_json = run_full_pipeline_entity_exact(clinical_cfg, seed=SEED)
print("Saved:", out_json)
!ls -lh $(dirname "$out_json")


In [ ]:
SEED = 42
out_json = run_full_pipeline_entity_exact(roberta_cfg, seed=SEED)
print("Saved:", out_json)
!ls -lh $(dirname "$out_json")


In [ ]:
SEED = 42
out_json = run_full_pipeline_entity_exact(biobert_cfg, seed=SEED)

print("Saved:", out_json)


In [ ]:
SEED = 42
out_json = run_full_pipeline_token_typed(clinical_cfg, seed=SEED)
print("Saved:", out_json)
!ls -lh $(dirname "$out_json")


In [ ]:
SEED = 42
out_json = run_full_pipeline_token_typed(roberta_cfg, seed=SEED)
print("Saved:", out_json)
!ls -lh $(dirname "$out_json")


In [ ]:
SEED = 42
out_json = run_full_pipeline_token_typed(biobert_cfg, seed=SEED)
print("Saved:", out_json)
!ls -lh $(dirname "$out_json")
